# 第 13 章　深度学习在金融中的应用

::: {.callout-note}
## 本章要点

1. **能力边界**：深度学习适合什么、不适合什么
2. **LSTM/GRU**：时序预测，与 ARIMA/XGBoost 对比
3. **Transformer 简介**：Informer / PatchTST 的核心改进直觉
4. **实战**：LSTM 预测板块指数收益率，多基准对比

本章建议在 Google Colab 运行（免费 GPU）。
:::


In [ ]:
# ── 第 13 章　深度学习　──────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
OUTPUT = 'output'; os.makedirs(OUTPUT, exist_ok=True)
RNG = np.random.default_rng(42)

# 检查 PyTorch
try:
    import torch
    print(f'PyTorch {torch.__version__}  GPU: {torch.cuda.is_available()}')
except ImportError:
    print('PyTorch 未安装。运行：pip install torch')
    print('建议在 Google Colab 运行本章（免费 GPU）')


---

## 1　能力边界：深度学习适合什么

| 场景 | DL 适合吗 | 推荐替代 |
|------|-----------|----------|
| 短期收益率预测（日度）| ⚠️ 市场效率高，信噪比极低 | ARIMA、ML 线性模型 |
| 中低频因子构造 | ✓ 非线性特征提取 | LSTM + 自注意力 |
| 文本情感分析 | ✓ NLP 天然适合 DL | BERT / FinBERT |
| 时序异常检测 | ✓ | Autoencoder、LSTM |
| 表格财务数据预测 | ✗ 通常不如树模型 | XGBoost、LightGBM |

::: {.callout-warning}
## 常见误区：DL 不是万能的

在金融预测任务的标准 Benchmark 中，
精心调参的 LightGBM 在绝大多数表格数据任务上不差于甚至优于 LSTM。
DL 的优势体现在：**原始数据有丰富空间/时序结构**（图像、文本、超高频 tick 数据）。
:::


In [ ]:
# ── 2.1  数据准备：模拟板块指数日度收益率 ──────────────────────────
T_dl  = 1000
# AR(1) + 季节性 + 噪声
rets_dl = np.zeros(T_dl)
for t in range(1, T_dl):
    rets_dl[t] = (0.08 * rets_dl[t-1]
                  + 0.03 * np.sin(2*np.pi*t/252)
                  + RNG.normal(0, 0.012))

SEQ_LEN = 30   # 用过去 30 天预测下一天
X_dl = np.array([rets_dl[i:i+SEQ_LEN] for i in range(T_dl-SEQ_LEN)])
y_dl = rets_dl[SEQ_LEN:]

split = int(len(y_dl) * 0.7)
X_tr_dl, X_te_dl = X_dl[:split], X_dl[split:]
y_tr_dl, y_te_dl = y_dl[:split], y_dl[split:]
print(f'训练集：{len(y_tr_dl)}  测试集：{len(y_te_dl)}')


In [ ]:
# ── 2.2  LSTM 模型定义与训练 ─────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_OK = True
except ImportError:
    TORCH_OK = False

if TORCH_OK:
    class LSTMModel(nn.Module):
        def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                                batch_first=True, dropout=dropout)
            self.fc   = nn.Linear(hidden_size, 1)

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :]).squeeze(-1)

    # 转为 Tensor（增加特征维度）
    Xtr_t = torch.FloatTensor(X_tr_dl[:, :, None])
    ytr_t = torch.FloatTensor(y_tr_dl)
    Xte_t = torch.FloatTensor(X_te_dl[:, :, None])

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=64, shuffle=True)
    model_lstm = LSTMModel()
    optimizer  = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)
    criterion  = nn.MSELoss()

    model_lstm.train()
    for epoch in range(30):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model_lstm(xb), yb)
            loss.backward()
            optimizer.step()
    print('LSTM 训练完成（30 epochs）')

    model_lstm.eval()
    with torch.no_grad():
        pred_lstm = model_lstm(Xte_t).numpy()
else:
    # 用简单 AR 模拟 LSTM 输出（无 PyTorch 时的退化方案）
    from sklearn.linear_model import Ridge
    m_ridge = Ridge().fit(X_tr_dl, y_tr_dl)
    pred_lstm = m_ridge.predict(X_te_dl)
    print('（PyTorch 未安装，用 Ridge 代替 LSTM 演示）')


In [ ]:
# ── 2.3  基准模型对比：ARIMA / Ridge / XGBoost / LSTM ───────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb

# Ridge 基准
from sklearn.linear_model import Ridge as Rdg
pred_ridge = Rdg().fit(X_tr_dl, y_tr_dl).predict(X_te_dl)

# XGBoost 基准
pred_xgb = xgb.XGBRegressor(n_estimators=200, max_depth=3,
                              learning_rate=0.05, random_state=42,
                              verbosity=0).fit(X_tr_dl, y_tr_dl).predict(X_te_dl)

# 朴素基准（预测均值）
pred_naive = np.full(len(y_te_dl), y_tr_dl.mean())

print('模型对比（测试集）：')
print(f"{'模型':<12} {'RMSE':>10} {'MAE':>10}")
print('─' * 34)
for name, pred in [('朴素均值', pred_naive), ('Ridge', pred_ridge),
                    ('XGBoost', pred_xgb), ('LSTM', pred_lstm)]:
    rmse = np.sqrt(mean_squared_error(y_te_dl, pred))
    mae  = mean_absolute_error(y_te_dl, pred)
    print(f'{name:<12} {rmse:>10.6f} {mae:>10.6f}')

print()
print('注：金融时序预测的信噪比极低，LSTM 不一定优于简单模型。')
print('评估重点应放在「方向准确率」和「策略收益」，而非 RMSE。')


---

## 3　Transformer 时序预测简介

### 3.1　标准 Transformer 的时序问题

Transformer（Vaswani et al., 2017）的自注意力机制
复杂度为 $O(L^2)$，当序列长度 $L$ 很大（如日度数据几年）时计算量爆炸。

**两种改进方向：**

| 模型 | 核心改进 | 适用场景 |
|------|----------|----------|
| **Informer**（Zhou et al., 2021）| ProbSparse 稀疏注意力，复杂度降至 $O(L \log L)$ | 长序列预测（日度→月度）|
| **PatchTST**（Nie et al., 2023）| 把时序切分为 Patch，用 ViT 风格处理 | 多变量时序，局部模式提取 |

### 3.2　实际使用建议

- 对于日度收益率预测：树模型或 LSTM 通常够用且更稳定
- Transformer 在**长期预测**（30 天 + 的宏观指标）和**多变量复杂依赖**场景更有优势
- `pytorch-forecasting` 库提供了开箱即用的 TemporalFusionTransformer

::: {.callout-tip}
## 提示词示范（LSTM 时序预测）

````
我有一个股票日度收益率时序 DataFrame，列为 date 和 ret。
请帮我：
1. 用过去 30 天的收益率作为特征，预测下一天的收益率
2. 用 70/30 的比例划分训练/测试集（不随机打乱）
3. 实现一个 2 层 LSTM（hidden_size=64, dropout=0.2），
   训练 50 epochs，batch_size=32，优化器 Adam(lr=1e-3)
4. 与 Ridge 和 XGBoost 基准模型比较 RMSE 和 MAE
5. 绘制测试集上的「实际 vs 预测」折线图，
   同时标注方向准确率（预测方向与实际方向一致的比例）
````
:::


---

## 4　章末练习

**练习 1（LSTM 调参）**
用 `hidden_size` ∈ {32, 64, 128} 和 `num_layers` ∈ {1, 2} 做网格搜索，
用验证集（训练集的最后 20%）选择最优超参数，报告测试集 RMSE。

**练习 2（方向准确率）**
将回归问题转换为分类问题：预测明日收益率正负。
分别评估 LSTM、XGBoost 的方向准确率（AUC），
讨论为什么方向准确率比 RMSE 对量化策略更有意义。

**练习 3（Colab GPU）**
把本章代码上传到 Google Colab，
切换到 GPU 运行时，比较 CPU 和 GPU 的训练时间差异。

**练习 4（思考题）**
有研究者声称他的 LSTM 模型在日度股票收益率预测上达到了 60% 的方向准确率。
你需要做哪些检验来判断这个结果是否可信？
（至少列出 3 个可能存在的问题。）
